## Setup Inicial

In [1]:
#%pip install nltk spacy gensim matplotlib scikit-learn
import nltk
import spacy
import string
import math
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from nltk.corpus import stopwords, wordnet as wn
from nltk.stem import RSLPStemmer
from nltk.tokenize import word_tokenize

# Download dos recursos necessários
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Download do modelo spaCy para português
#!python -m spacy download pt_core_news_sm

# Carregar modelo spaCy
nlp = spacy.load("pt_core_news_sm")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


---
## Parte 1 - VSM e Similaridade de Cosseno

In [2]:
documentos = [ 
    "O gato preto dorme no sofá da sala", 
    "O cachorro late para o gato preto", 
    "Gatos e cachorros são animais domésticos", 
    "A sala tem um sofá confortável", 
    "Animais selvagens vivem na floresta" 
] 
 
query = "gato preto no sofá" 

### Questão 1.1 - Pré-processamento

In [3]:
stop_words = set(stopwords.words('portuguese'))

def preprocess_text(texto):
    """
    Recebe um texto e retorna a lista de tokens pré-processados.
    
    Dicas: 
    1. Use word_tokenize(texto.lower(), language='portuguese') 
    2. Filtre tokens que não estão em stop_words e não são pontuação 
    3. string.punctuation contém os caracteres de pontuação
    
    >>> preprocess_text("O gato dorme!")
    ['gato', 'dorme']
    """
    # 1. Tokenizar o texto em minúsculas
    tokens = word_tokenize(texto.lower(), language='portuguese')
    
    # 2. Remover stopwords e pontuação
    tokens_processados = [
        t for t in tokens
        if t not in stop_words and t not in string.punctuation
    ]
    
    return tokens_processados

# Teste
print("Teste da função:")
print(preprocess_text("O gato preto dorme no sofá!"))
print()

# Processar documentos
documentos_processados = [preprocess_text(doc) for doc in documentos]
query_processada = preprocess_text(query)

print("Documentos processados:")
for i, doc in enumerate(documentos_processados):
    print(f"Doc {i+1}: {doc}")
print(f"Query processada: {query_processada}")

Teste da função:
['gato', 'preto', 'dorme', 'sofá']

Documentos processados:
Doc 1: ['gato', 'preto', 'dorme', 'sofá', 'sala']
Doc 2: ['cachorro', 'late', 'gato', 'preto']
Doc 3: ['gatos', 'cachorros', 'animais', 'domésticos']
Doc 4: ['sala', 'sofá', 'confortável']
Doc 5: ['animais', 'selvagens', 'vivem', 'floresta']
Query processada: ['gato', 'preto', 'sofá']


### Questão 1.2 - Implemente manualmente o cálculo de TF-IDF

In [4]:
def compute_tf(tokens): 
    """ 
    Term Frequency: frequência do termo no documento 
    Fórmula: TF(t,d) = frequência do termo t / total de termos em d 
     
    Dica: Use Counter(tokens) para contar frequências 
    """
    total = len(tokens)
    freq = Counter(tokens)
    
    tf = {termo: count / total for termo, count in freq.items()}
    
    return tf


def compute_idf(documentos_processados): 
    """ 
    Inverse Document Frequency: log(N / df) 
    Onde: 
    - N = número total de documentos 
    - df = número de documentos que contêm o termo 
     
    Dica: Use set(doc) para obter termos únicos por documento 
    """ 
    N = len(documentos_processados)
    
    # Calcular document frequency (df)
    df = {}
    for doc in documentos_processados:
        for termo in set(doc):  # set() garante que cada termo conta 1x por doc
            df[termo] = df.get(termo, 0) + 1
    
    # Calcular IDF para cada termo
    idf = {}
    for termo, freq in df.items():
        idf[termo] = math.log(N / freq)
    
    return idf

def compute_tfidf(doc_tokens, idf_dict): 
    """ 
    Calcula TF-IDF para um documento 
    TF-IDF(t,d) = TF(t,d) * IDF(t) 
    """
    tf = compute_tf(doc_tokens)
    
    tfidf = {}
    for termo, tf_val in tf.items():
        if termo in idf_dict:
            tfidf[termo] = tf_val * idf_dict[termo]
    
    return tfidf


# Calcular IDF
idf = compute_idf(documentos_processados)
print("IDF (termos relevantes):")
termos_mostrar = ['gato', 'preto', 'sofá', 'sala', 'animais']
for termo in termos_mostrar:
    if termo in idf:
        print(f"  {termo}: {idf[termo]:.4f}")
print()

# Calcular vetores TF-IDF
vetores_docs = [compute_tfidf(doc, idf) for doc in documentos_processados]
vetor_query = compute_tfidf(query_processada, idf)

print("Vetor da query:")
print(vetor_query)

IDF (termos relevantes):
  gato: 0.9163
  preto: 0.9163
  sofá: 0.9163
  sala: 0.9163
  animais: 0.9163

Vetor da query:
{'gato': 0.3054302439580517, 'preto': 0.3054302439580517, 'sofá': 0.3054302439580517}


### Questão 1.3 -  Implemente a função de similaridade de cosseno para calcular a similaridade entre a query e cada documento.

In [5]:
def cosine_similarity(v1, v2): 
    """ 
    Calcula a similaridade de cosseno entre dois vetores esparsos. 
     
    Fórmula: cos(θ) = (v1 · v2) / (||v1|| * ||v2||) 
     
    Passos: 
    1. Encontrar termos comuns (intersecção das chaves) 
    2. Calcular produto escalar (soma dos produtos dos valores comuns) 
    3. Calcular normas (raiz quadrada da soma dos quadrados) 
    4. Dividir produto escalar pelo produto das normas 
     
    Dica: Use set(v1.keys()) & set(v2.keys()) para termos comuns 
    """ 
    # Encontrar termos comuns
    termos_comuns = set(v1.keys()) & set(v2.keys())
    
    # Calcular produto escalar
    dot_product = sum(v1[t] * v2[t] for t in termos_comuns)
    
    # Calcular normas
    norm1 = math.sqrt(sum(val**2 for val in v1.values()))
    norm2 = math.sqrt(sum(val**2 for val in v2.values()))
    
    # Evitar divisão por zero
    if norm1 == 0 or norm2 == 0:
        return 0
    
    # Calcular similaridade
    similaridade = dot_product / (norm1 * norm2)
    
    return similaridade


# Calcular similaridades
print("Similaridades Query vs Documentos:")
print("-" * 50)

similaridades = []
for i, doc_vetor in enumerate(vetores_docs):
    sim = cosine_similarity(vetor_query, doc_vetor)
    similaridades.append((i, sim))
    print(f"Doc {i+1}: {sim:.4f} - '{documentos[i]}'")

# Ordenar documentos por similaridade (decrescente)
similaridades_ordenadas = sorted(similaridades, key=lambda x: x[1], reverse=True)

print("\n" + "="*50)
print("RANKING DE RELEVÂNCIA:")
print("="*50)
for idx, sim in similaridades_ordenadas:
    if sim > 0:
        print(f"{sim:.4f} - Doc {idx+1}: {documentos[idx]}")

Similaridades Query vs Documentos:
--------------------------------------------------
Doc 1: 0.6507 - 'O gato preto dorme no sofá da sala'
Doc 2: 0.4040 - 'O cachorro late para o gato preto'
Doc 3: 0.0000 - 'Gatos e cachorros são animais domésticos'
Doc 4: 0.2560 - 'A sala tem um sofá confortável'
Doc 5: 0.0000 - 'Animais selvagens vivem na floresta'

RANKING DE RELEVÂNCIA:
0.6507 - Doc 1: O gato preto dorme no sofá da sala
0.4040 - Doc 2: O cachorro late para o gato preto
0.2560 - Doc 4: A sala tem um sofá confortável


### Questão 1.4 - Como é que a remoção de stopwords afetou os resultados?

A remoção de stopwords permitiu focar o modelo apenas nas palavras com conteúdo semântico relevante (e.g., "gato", "sofá"), eliminando termos muito frequentes como "o", "a", "no", "da" que aparecem em todos os documentos e que não ajudam a distinguir os seus tópicos. Sem esta remoção, palavras como "o" teriam um IDF próximo de zero (aparecem em quase todos os docs), contribuindo pouco para a similaridade mas adicionando ruído ao vetor. Com stopwords removidas, os documentos que partilham palavras conteúdo com a query (Doc 1: "gato", "preto", "sofá"; Doc 4: "sala", "sofá") ficam claramente mais relevantes.

### Questão 1.5 - Duas limitações do modelo TF-IDF + Cosseno

1. **Ignora a semântica/sinónimos**: "cachorro" e "cão" são tratados como palavras completamente diferentes, mesmo significando o mesmo. O modelo não consegue reconhecer que Doc 2 também fala de um "gato" no mesmo contexto de Doc 1.

2. **Ignora a ordem das palavras**: Os documentos "O gato mordeu o cão" e "O cão mordeu o gato" teriam exatamente o mesmo vetor TF-IDF, apesar de terem significados opostos.

---
## Parte 2 - WordNet - Explorando Relações Semânticas

### Questão 2.1 - Complete o bloco de código de forma a explorar os synsets de uma palavra e as suas relações. 

In [6]:
def explorar_palavra(palavra, lang='por'): 
    """ 
    Explora os synsets de uma palavra e as suas relações. 
     
    Passos: 
    1. Use wn.synsets(palavra, lang=lang) para obter synsets 
    2. Para cada synset, mostre: 
       - Nome: syn.name() 
       - Definição: syn.definition() 
       - Exemplos: syn.examples() 
       - Lemmas (palavras): [l.name() for l in syn.lemmas()] 
       - Hiperónimos: syn.hypernyms() 
       - Hipónimos: syn.hyponyms() 
    """
    
    synsets = wn.synsets(palavra, lang=lang)
    
    if not synsets:
        print(f"Palavra '{palavra}' não encontrada")
        return
    
    print(f"\n{'='*50}")
    print(f"Explorando: {palavra}") 
    print(f"Número de synsets: {len(synsets)}")
    
    #para cada synset
    for i, syn in enumerate(synsets[:3]):  # Mostrar apenas 3
        print(f"\n--- Synset {i+1}: {syn.name()} ---")
        
        # definição
        print(f"  Definição: {syn.definition()}")
        
        # exemplos 
        if syn.examples():
            print(f"  Exemplos: {syn.examples()}")
        
        #lemmas (sinónimos)
        lemmas = [l.name() for l in syn.lemmas()]
        print(f"  Lemmas: {lemmas}")
        
        # hiperónimos 
        hiper = syn.hypernyms()
        if hiper:
            print(f"  Hiperónimos: {[h.name() for h in hiper]}")
        
        # hipónimos
        hipo = syn.hyponyms()
        if hipo:
            print(f"  Hipónimos (primeiros 3): {[h.name() for h in hipo[:3]]}") # apenas 3


# palavras de teste
palavras_teste = ['cachorro', 'computador', 'casa', 'amor', 'correr']
for palavra in palavras_teste:
    explorar_palavra(palavra)


Explorando: cachorro
Número de synsets: 2

--- Synset 1: bitch.n.04 ---
  Definição: female of any member of the dog family
  Lemmas: ['bitch']
  Hiperónimos: ['canine.n.02']
  Hipónimos (primeiros 3): ['brood_bitch.n.01']

--- Synset 2: dog.n.01 ---
  Definição: a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
  Exemplos: ['the dog barked all night']
  Lemmas: ['dog', 'domestic_dog', 'Canis_familiaris']
  Hiperónimos: ['canine.n.02', 'domestic_animal.n.01']
  Hipónimos (primeiros 3): ['great_pyrenees.n.01', 'working_dog.n.01', 'hunting_dog.n.01']

Explorando: computador
Número de synsets: 3

--- Synset 1: calculator.n.02 ---
  Definição: a small machine that is used for mathematical calculations
  Lemmas: ['calculator', 'calculating_machine']
  Hiperónimos: ['machine.n.01']
  Hipónimos (primeiros 3): ['abacus.n.02', "napier's_bones.n.01", 'counter.n.03']

--- Synset 2: computer.n.01 

### Questão 2.2 - Similaridade WordNet (path similarity)

In [7]:
def similaridade_wordnet(palavra1, palavra2, lang='por'): 
    """ 
    Calcula similaridade entre palavras usando caminho no WordNet. 
     
    path_similarity: 1 / (distância + 1) 
     
    Passos: 
    1. Obter synsets de ambas as palavras 
    2. Para cada par de synsets, calcular path_similarity 
    3. Retornar a maior similaridade encontrada 
    """ 
    # Obter synsets
    syns1 = wn.synsets(palavra1, lang=lang)
    syns2 = wn.synsets(palavra2, lang=lang)
    
    if not syns1 or not syns2:
        return None, None
    
    max_sim = 0
    melhor_par = None
    
    # Comparar todos os pares
    for s1 in syns1[:3]:  # Limitar para performance
        for s2 in syns2[:3]:
            try:
                # Calcular similaridade
                sim = s1.path_similarity(s2)
                
                # Atualizar máximo se necessário
                if sim is not None and sim > max_sim:
                    max_sim = sim
                    melhor_par = (s1.name(), s2.name())
            except:
                continue
    
    return max_sim, melhor_par


# Testar pares de palavras
pares = [
    ('cachorro', 'cão'),
    ('cachorro', 'gato'),
    ('cachorro', 'carro'),
    ('casa', 'residência'),
    ('casa', 'apartamento'),
    ('amor', 'ódio')
]

print("SIMILARIDADE SEMÂNTICA (WordNet)")
print("="*50)

for p1, p2 in pares:
    sim, par = similaridade_wordnet(p1, p2)
    if sim and sim > 0:
        print(f"{p1} vs {p2}: {sim:.4f}  (melhor par: {par})")
    else:
        print(f"{p1} vs {p2}: não foi possível calcular")

SIMILARIDADE SEMÂNTICA (WordNet)
cachorro vs cão: 1.0000  (melhor par: ('bitch.n.04', 'bitch.n.04'))
cachorro vs gato: 0.2500  (melhor par: ('dog.n.01', 'kitty.n.04'))
cachorro vs carro: 0.0909  (melhor par: ('dog.n.01', 'car.n.02'))
casa vs residência: 0.3333  (melhor par: ('apartment.n.01', 'dwelling.n.01'))
casa vs apartamento: 1.0000  (melhor par: ('apartment.n.01', 'apartment.n.01'))
amor vs ódio: 0.1250  (melhor par: ('affectionateness.n.02', 'antipathy.n.01'))


---
## Parte 3 - Dependency Parsing com spaCy

### Questão 3.1 - Complete o código de forma a conseguir analisar as frases de teste e mostrar as suas dependências.

In [8]:
def analisar_frase(frase): 
    """ 
    Analisa uma frase e mostra as suas dependências. 
     
    Passos: 
    1. Processar frase com nlp(frase) 
    2. Para cada token, mostrar: 
       - token.text: a palavra 
       - token.pos_: part-of-speech tag 
       - token.dep_: relação de dependência 
       - token.head.text: o governador (palavra da qual depende) 
    """
    # Processar frase com spaCy
    doc = nlp(frase)
    
    print(f"\nFrase: '{frase}'")
    print("-" * 60)
    print(f"{'Token':<15} {'POS':<10} {'Dependência':<15} {'Governador':<15}")
    print("-" * 60)
    
    # Iterar sobre tokens e mostrar informações
    for token in doc:
        print(f"{token.text:<15} {token.pos_:<10} {token.dep_:<15} {token.head.text:<15}")
    
    return doc


# Testar com frases
frases_teste = [
    "O cachorro mordeu o homem",
    "A menina de vestido azul comeu um bolo delicioso"
]

for frase in frases_teste:
    analisar_frase(frase)


Frase: 'O cachorro mordeu o homem'
------------------------------------------------------------
Token           POS        Dependência     Governador     
------------------------------------------------------------
O               DET        det             cachorro       
cachorro        NOUN       nsubj           mordeu         
mordeu          VERB       ROOT            mordeu         
o               DET        det             homem          
homem           NOUN       obj             mordeu         

Frase: 'A menina de vestido azul comeu um bolo delicioso'
------------------------------------------------------------
Token           POS        Dependência     Governador     
------------------------------------------------------------
A               DET        det             menina         
menina          NOUN       nsubj           comeu          
de              ADP        case            vestido        
vestido         NOUN       nmod            menina         
azul        

### Questão 3.2 - Completa o código de forma a extrair relações sujeito-verbo-objeto de um documento. 

In [9]:
def extrair_relacoes(doc): 
    """ 
    Extrai relações sujeito-verbo-objeto de um documento. 
     
    Dica:  
    - Verbos têm token.pos_ == "VERB" 
    - Sujeitos têm token.dep_ in ["nsubj", "nsubj:pass"] 
    - Objetos têm token.dep_ in ["obj", "iobj", "obl"] 
    - Use token.children para acessar os dependentes 
    """
    relacoes = []
    
    # Encontrar verbos e as suas relações
    for token in doc:
        if token.pos_ == "VERB":
            verbo = token.text
            
            # Encontrar sujeitos (filhos com dep_ de sujeito)
            sujeitos = [
                child for child in token.children
                if child.dep_ in ["nsubj", "nsubj:pass"]
            ]
            
            # Encontrar objetos (filhos com dep_ de objeto)
            objetos = [
                child for child in token.children
                if child.dep_ in ["obj", "iobj", "obl"]
            ]
            
            # Criar pares sujeito-verbo-objeto
            for suj in sujeitos:
                for obj in objetos:
                    relacao = {
                        'sujeito': suj.text,
                        'verbo': verbo,
                        'objeto': obj.text
                    }
                    relacoes.append(relacao)
                    print(f"  {suj.text} → {verbo} → {obj.text}")
    
    return relacoes


def extrair_modificadores(doc): 
    """ 
    Extrai modificadores (adjetivos ligados a substantivos). 
     
    Dica:  
    - Adjetivos têm token.pos_ == "ADJ" 
    - O substantivo modificado é token.head se head.pos_ for "NOUN" 
    """ 
    modificadores = []
    
    # Encontrar adjetivos e o substantivo que modificam
    for token in doc:
        if token.pos_ == "ADJ":
            # Verificar se o governador é um substantivo
            substantivo = token.head if token.head.pos_ == "NOUN" else None
            
            if substantivo:
                modificador = {
                    'substantivo': substantivo.text,
                    'adjetivo': token.text
                }
                modificadores.append(modificador)
                print(f"  {substantivo.text} é {token.text}")
    
    return modificadores


# Teste
frase = "O carro vermelho e rápido ultrapassou o caminhão lento"
doc = nlp(frase)

print("Relações Sujeito-Verbo-Objeto:")
relacoes = extrair_relacoes(doc)

print("\nModificadores:")
modificadores = extrair_modificadores(doc)

Relações Sujeito-Verbo-Objeto:
  carro → ultrapassou → caminhão

Modificadores:
  carro é vermelho
  caminhão é lento
